# RC9.2.1 — DEEP 14,400 s, run as resumable segments
## Day-2 follow-up (SAMPLE)

This is the **second** package. Run it only where the 2,700 s pass says a full
budget is needed, or on the scenarios you care most about.

### Why segments and not one 4-hour run

A four-hour unattended run does not survive a Colab disconnect. It did not
survive in our own container either — two attempts were killed at 10 and 40
minutes. So the budget is reached as **6 segments of 2,400 s**, each resuming
the last from the engine's own checkpoints. A lost segment costs one segment.

### Resume only started working today

`--resume` was broken in every RC9.2.1 build before this package. `run_id` was
hashed from `run_parameters`, which contained an absolute wall-clock timestamp,
so the recomputed identity never matched the checkpoint and resume always raised
*"Resume refused"*. That is fixed here (finding **A32**), and the engine in this
package is the fixed one — check `SCENARIOS.json` → `engine_sha256`.

**One scenario per Colab instance.** Do not shard scenarios inside one instance
at this budget.

### Variant A: no Google Drive

Results stay in the VM. A disconnect loses them, so **variant B is strongly preferred at this budget**.

## Changed after run 1: ONE session, not six segments

Segmenting **does not reach the target budget**. Each segment re-plans its phases for the segment length, so Stage 1 gets the same small window every time and the 15-profile portfolio never completes. Run 1 spent roughly 15 hours of Colab compute this way and finished no portfolio on any scenario.

Defaults are now `SEGMENTS = 1`, `SEGMENT_SEC = 14400` — the whole budget in one run. **Keep the tab open**; a running cell is not idle, so a four-hour run holds a session. Use the Drive variant so a disconnect costs the run and not the results.


In [ ]:
#@title 1. Environment { display-mode: "form" }
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" 2>&1 | tail -2
import multiprocessing, platform
print("cpus:", multiprocessing.cpu_count(), "| python:", platform.python_version())
import ortools; print("ortools:", ortools.__version__)

In [ ]:
#@title 2. Upload the package ZIP { display-mode: "form" }
from google.colab import files
import zipfile, io, os
os.makedirs("/content/rc921deep", exist_ok=True)
up = files.upload()
with zipfile.ZipFile(io.BytesIO(up[next(iter(up))])) as z:
    z.extractall("/content/rc921deep")
print("extracted")

In [ ]:
#@title 3. Locate the package { display-mode: "form" }
import pathlib
c = list(pathlib.Path("/content/rc921deep").rglob("SCENARIOS.json"))
assert c, "SCENARIOS.json not found"
ROOT = c[0].parent
RESULTS_ROOT = ROOT / "results"; RESULTS_ROOT.mkdir(exist_ok=True)
import json; m = json.loads((ROOT/"SCENARIOS.json").read_text())
print("engine:", m["engine_release"]); print("sha256:", m["engine_sha256"][:32], "…")
print("ROOT =", ROOT)

In [ ]:
#@title 4. Run the segments { display-mode: "form" }
SCENARIO      = "CRICUT_VOICE"  #@param ["NMG_SP","CRICUT_VOICE","CRICUT_CHAT","AE_AR_B2B","GDI_REAL28","NMG_EN_SP","NMG_EN"]
SEGMENT_SEC   = 14400  #@param {type:"integer"}
SEGMENTS      = 1     #@param {type:"integer"}
CONTINUE_FROM = 1     #@param {type:"integer"}

import subprocess, sys, multiprocessing
cmd = [sys.executable, "-u", str(ROOT/"runners"/"rc921_deep_segmented.py"),
       "--package-root", str(ROOT), "--results-root", str(RESULTS_ROOT),
       "--only", SCENARIO, "--segment-sec", str(SEGMENT_SEC),
       "--segments", str(SEGMENTS), "--continue-from", str(CONTINUE_FROM),
       "--num-workers", str(multiprocessing.cpu_count())]
print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout: print(line, end="")
print("\nexit:", proc.wait())

## 5. If Colab disconnects mid-run

Re-open the notebook, re-run cells 1–3, then set **`CONTINUE_FROM`** to the
segment that did not finish and run cell 4 again. Completed segments are already
banked in the engine's checkpoints.

`<SCENARIO>_SEGMENT_LEDGER.json` in the results directory records which segments
ran and how many skeletons were banked after each — read it to find where to
resume.

In [ ]:
#@title 6. Download results { display-mode: "form" }
import shutil, time, os
from google.colab import files
out = f"/content/RC921_DEEP_{SCENARIO}_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(out, "zip", str(RESULTS_ROOT))
print(round(os.path.getsize(out+".zip")/1e6,1), "MB"); files.download(out+".zip")